# Pandas 极简入门：上市公司数据

这一节用两张小表介绍 pandas 的基本功能：

- `company_info_example.xlsx`：公司基本信息，每家公司一行。
- `financial_data_example.xlsx`：年度财务数据，每家公司每年一行。

这两张表来自真实上市公司数据的抽样，并故意保留了少量常见问题：证券代码格式、缺失值、重复行、数字列混入文本等。

## 1. 读取数据

证券代码是编号，不是可以计算大小的数字。读取时先把它作为字符串处理，并补齐到 6 位。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)

def read_code(x):
    return str(x).strip().zfill(6)

company = pd.read_excel(
    "data/company_info_example.xlsx",
    converters={"证券代码": read_code},
)

finance = pd.read_excel(
    "data/financial_data_example.xlsx",
    converters={"证券代码": read_code},
)

company.head()

,证券代码,证券简称,公司全称,上市市场,行业代码,行业名称,省份,城市,上市日期,成立日期,上市状态
0,000001,平安银行,平安银行股份有限公司,SZSE,J66,货币金融服务,广东省,深圳市,1991-04-03,1987-12-22,正常上市
1,000002,万科A,万科企业股份有限公司,SZSE,K70,房地产业,广东省,深圳市,1991-01-29,1988-11-01,正常上市
2,000004,国华网安,深圳国华网安科技股份有限公司,SZSE,I65,软件和信息技术服务业,广东省,深圳市,1991-01-14,1986-05-05,正常上市
3,000006,深振业A,深圳市振业(集团)股份有限公司,SZSE,K70,NaN,广东省,深圳市,1992-04-27,1989-04-01,正常上市
4,000007,*ST 全新,深圳市全新好股份有限公司,SZSE,K70,房地产业,广东省,深圳市,1992-04-13,1988-11-21,ST


In [2]:
finance.head()

,证券代码,证券简称,统计截止日期,年份,总资产,总负债,净资产,营业收入,净利润,资产负债率
0,000001,平安银行,2018-12-31,2018,3418592000000,3.178550e+12,2.400420e+11,1.062120e+11,24818000000,0.9298
1,000001,平安银行,2019-12-31,2019,3939070000000,3.626087e+12,3.129830e+11,1.268140e+11,28195000000,0.9205
2,000001,平安银行,2020-12-31,2020,4468514000000,4.104383e+12,3.641310e+11,1.432420e+11,28928000000,0.9185
3,000002,万科A,2018-12-31,2018,1528579356474.810059,1.292959e+12,2.356207e+11,2.976793e+11,49272294534.610001,0.8459
4,000002,万科A,2019-12-31,2019,1729929450401.22998,1.459350e+12,2.705791e+11,3.678939e+11,55131614572.089996,0.8436


## 2. 先看数据

拿到一张表后，通常先看行列数、列名、数据类型、缺失值和几个样本行。

In [3]:
print(company.shape)
print(company.columns.tolist())

company.info()

(30, 11)
['证券代码', '证券简称', '公司全称', '上市市场', '行业代码', '行业名称', '省份', '城市', '上市日期', '成立日期', '上市状态']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   证券代码    30 non-null     object
 1   证券简称    30 non-null     object
 2   公司全称    30 non-null     object
 3   上市市场    30 non-null     object
 4   行业代码    30 non-null     object
 5   行业名称    29 non-null     object
 6   省份      30 non-null     object
 7   城市      29 non-null     object
 8   上市日期    30 non-null     object
 9   成立日期    30 non-null     object
 10  上市状态    30 non-null     object
dtypes: object(11)
memory usage: 2.7+ KB


In [4]:
print(finance.shape)
print(finance.dtypes)

finance.isna().sum()

(89, 10)
证券代码              object
证券简称              object
统计截止日期    datetime64[ns]
年份                 int64
总资产               object
总负债              float64
净资产              float64
营业收入             float64
净利润               object
资产负债率            float64
dtype: object


证券代码      0
证券简称      0
统计截止日期    0
年份        0
总资产       0
总负债       0
净资产       0
营业收入      1
净利润       0
资产负债率     0
dtype: int64

`value_counts()` 适合快速查看分类变量的分布。

In [5]:
company["行业名称"].value_counts(dropna=False).head(10)

行业名称
货币金融服务              4
软件和信息技术服务业          4
计算机、通信和其他电子设备制造业    4
汽车制造业               4
电气机械及器材制造业          4
医药制造业               4
房地产业                3
批发业                 2
NaN                 1
Name: count, dtype: int64

## 3. 选择行和列

`[]` 常用于选列，`.loc` 按行列标签选择，`.iloc` 按整数位置选择。

In [6]:
company[["证券代码", "证券简称", "行业名称", "省份"]].head()

,证券代码,证券简称,行业名称,省份
0,000001,平安银行,货币金融服务,广东省
1,000002,万科A,房地产业,广东省
2,000004,国华网安,软件和信息技术服务业,广东省
3,000006,深振业A,NaN,广东省
4,000007,*ST 全新,房地产业,广东省


In [7]:
company.loc[company["省份"] == "广东省", ["证券代码", "证券简称", "城市"]].head()

,证券代码,证券简称,城市
0,000001,平安银行,深圳市
1,000002,万科A,深圳市
2,000004,国华网安,深圳市
3,000006,深振业A,深圳市
4,000007,*ST 全新,深圳市


In [8]:
finance.iloc[:5, :5]

,证券代码,证券简称,统计截止日期,年份,总资产
0,000001,平安银行,2018-12-31,2018,3418592000000
1,000001,平安银行,2019-12-31,2019,3939070000000
2,000001,平安银行,2020-12-31,2020,4468514000000
3,000002,万科A,2018-12-31,2018,1528579356474.810059
4,000002,万科A,2019-12-31,2019,1729929450401.22998


## 4. 清洗常见脏数据

先处理文本两端空格、日期类型、缺失值、重复行，以及数字列中混入的文本。

In [9]:
company["证券简称"] = company["证券简称"].str.strip()
company["上市日期"] = pd.to_datetime(company["上市日期"], errors="coerce")
company["成立日期"] = pd.to_datetime(company["成立日期"], errors="coerce")

company[company["行业名称"].isna() | company["城市"].isna()]

,证券代码,证券简称,公司全称,上市市场,行业代码,行业名称,省份,城市,上市日期,成立日期,上市状态
3,000006,深振业A,深圳市振业(集团)股份有限公司,SZSE,K70,NaN,广东省,深圳市,1992-04-27,1989-04-01,正常上市
7,000019,深粮控股,深圳市深粮控股股份有限公司,SZSE,F51,批发业,广东省,NaN,1992-10-12,1992-08-06,正常上市


In [10]:
# 这里为了演示，行业用“未分类”填补，城市用“未知”填补。
company["行业名称"] = company["行业名称"].fillna("未分类")
company["城市"] = company["城市"].fillna("未知")

company.isna().sum()

证券代码    0
证券简称    0
公司全称    0
上市市场    0
行业代码    0
行业名称    0
省份      0
城市      0
上市日期    1
成立日期    0
上市状态    0
dtype: int64

In [11]:
# 财务表里有一行重复记录。
print("重复行数量：", finance.duplicated().sum())
finance = finance.drop_duplicates()

# 数字列里可能混入逗号、-- 等文本。先统一转成字符串清理，再转回数值。
num_cols = ["总资产", "总负债", "净资产", "营业收入", "净利润", "资产负债率"]
for col in num_cols:
    finance[col] = (
        finance[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"--": np.nan, "nan": np.nan})
    )
    finance[col] = pd.to_numeric(finance[col], errors="coerce")

finance[num_cols].isna().sum()

重复行数量： 1


总资产      0
总负债      0
净资产      0
营业收入     1
净利润      1
资产负债率    0
dtype: int64

## 5. 列运算和类型转换

Pandas 的常见用法是基于已有列生成新列。

In [12]:
finance["统计截止日期"] = pd.to_datetime(finance["统计截止日期"], errors="coerce")
finance["年份"] = finance["统计截止日期"].dt.year

finance["总资产_亿元"] = finance["总资产"] / 1e8
finance["营业收入_亿元"] = finance["营业收入"] / 1e8
finance["净利率"] = finance["净利润"] / finance["营业收入"]
finance["是否盈利"] = finance["净利润"] > 0

finance["负债率等级"] = pd.cut(
    finance["资产负债率"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["低", "中", "高"],
)

finance[["证券代码", "年份", "总资产_亿元", "营业收入_亿元", "净利率", "是否盈利", "负债率等级"]].head()

,证券代码,年份,总资产_亿元,营业收入_亿元,净利率,是否盈利,负债率等级
0,000001,2018,34185.920000,1062.120000,0.233665,True,高
1,000001,2019,39390.700000,1268.140000,0.222333,True,高
2,000001,2020,44685.140000,1432.420000,0.201952,True,高
3,000002,2018,15285.793565,2976.793311,0.165521,True,高
4,000002,2019,17299.294504,3678.938775,0.149857,True,高


## 6. 排序、筛选和分组统计

下面找出 2020 年营业收入最高的几家公司。

In [13]:
finance_2020 = finance[finance["年份"] == 2020]

finance_2020.sort_values("营业收入", ascending=False)[
    ["证券代码", "证券简称", "营业收入_亿元", "净利率"]
].head(10)

,证券代码,证券简称,营业收入_亿元,净利率
53,000333,美的集团,2857.097290,0.096274
56,000338,潍柴动力,1974.910929,0.057090
81,600000,浦发银行,1746.870000,0.337707
87,600016,民生银行,1667.770000,0.210473
2,000001,平安银行,1432.420000,0.201952
84,600015,华夏银行,927.170000,0.232622
20,000016,深康佳A,503.518366,0.010726
69,000550,江铃汽车,330.957337,0.016640
66,000538,云南白药,327.427668,0.168313
41,000049,德赛电池,193.978245,0.038163


`groupby()` 用于“按组计算”。例如按年份统计平均资产负债率、营业收入总额和盈利公司数量。

In [14]:
finance.groupby("年份").agg(
    平均资产负债率=("资产负债率", "mean"),
    营业收入合计_亿元=("营业收入_亿元", "sum"),
    盈利公司数=("是否盈利", "sum"),
)

,平均资产负债率,营业收入合计_亿元,盈利公司数
年份,,,
2018,0.579013,14173.283653,26
2019,0.566357,15836.956191,26
2020,0.555227,13004.611757,27


## 7. 合并两张表

公司基本信息表有行业和省份，财务表有年度指标。两张表可以按 `证券代码` 合并。

In [15]:
merged = finance.merge(
    company[["证券代码", "行业名称", "省份", "城市", "上市日期"]],
    on="证券代码",
    how="left",
)

merged.head()

,证券代码,证券简称,统计截止日期,年份,总资产,总负债,净资产,营业收入,净利润,资产负债率,总资产_亿元,营业收入_亿元,净利率,是否盈利,负债率等级,行业名称,省份,城市,上市日期
0,000001,平安银行,2018-12-31,2018,3.418592e+12,3.178550e+12,2.400420e+11,1.062120e+11,2.481800e+10,0.9298,34185.920000,1062.120000,0.233665,True,高,货币金融服务,广东省,深圳市,1991-04-03
1,000001,平安银行,2019-12-31,2019,3.939070e+12,3.626087e+12,3.129830e+11,1.268140e+11,2.819500e+10,0.9205,39390.700000,1268.140000,0.222333,True,高,货币金融服务,广东省,深圳市,1991-04-03
2,000001,平安银行,2020-12-31,2020,4.468514e+12,4.104383e+12,3.641310e+11,1.432420e+11,2.892800e+10,0.9185,44685.140000,1432.420000,0.201952,True,高,货币金融服务,广东省,深圳市,1991-04-03
3,000002,万科A,2018-12-31,2018,1.528579e+12,1.292959e+12,2.356207e+11,2.976793e+11,4.927229e+10,0.8459,15285.793565,2976.793311,0.165521,True,高,房地产业,广东省,深圳市,1991-01-29
4,000002,万科A,2019-12-31,2019,1.729929e+12,1.459350e+12,2.705791e+11,3.678939e+11,5.513161e+10,0.8436,17299.294504,3678.938775,0.149857,True,高,房地产业,广东省,深圳市,1991-01-29


In [16]:
industry_summary = (
    merged[merged["年份"] == 2020]
    .groupby("行业名称")
    .agg(
        公司数=("证券代码", "nunique"),
        平均营业收入_亿元=("营业收入_亿元", "mean"),
        平均净利率=("净利率", "mean"),
    )
    .sort_values("平均营业收入_亿元", ascending=False)
)

industry_summary

,公司数,平均营业收入_亿元,平均净利率
行业名称,,,
货币金融服务,4,1443.557500,0.245688
电气机械及器材制造业,3,1032.766606,0.048928
汽车制造业,4,631.454842,0.049060
计算机、通信和其他电子设备制造业,4,140.555926,0.019264
医药制造业,4,124.984490,0.103415
批发业,2,61.544734,0.088032
软件和信息技术服务业,4,52.403242,0.022898
未分类,1,29.347333,0.307657
房地产业,3,20.747605,0.178185


## 8. 保存结果

清洗和合并后的结果可以保存为 Excel 或 CSV。一般保存普通表格时使用 `index=False`。

In [17]:
merged.to_excel("data/merged_finance_company.xlsx", index=False)
print("已保存：data/merged_finance_company.xlsx")

已保存：data/merged_finance_company.xlsx
